# Virtual Yoga Trainer Using MoveNet - Colab notebook

This notebook runs the **same core modules** (`ai/`, `utils/`, `database/`) that the deployed Streamlit app uses,
so everything you test here behaves identically in the web app.

**Pipeline:** Camera/Video -> MoveNet -> 17 keypoints -> normalisation -> smoothing -> joint angles -> pose classification
-> alignment analysis -> explainable correction -> scoring -> stability -> hold timer -> session history -> personalisation -> recommendations.

> The pose classifier is a **transparent rule-based (geometric) classifier**, not a trained ML model, and no accuracy figure is claimed.

**Run the cells from top to bottom.** Sections 5-7 need a video or an image (upload it or use the webcam snapshot).

## 1. Get the project code

In [ ]:
import os, sys, subprocess, glob, zipfile

# Option A: clone your GitHub repository (replace YOUR-USERNAME), or
# Option B: leave the placeholder and upload ai-yoga-trainer.zip when asked.
REPO_URL = "https://github.com/YOUR-USERNAME/ai-yoga-trainer.git"  #@param {type:"string"}

def find_root():
    """Return the project root (folder that contains ai/pipeline.py), or None."""
    for base in (".", "..", "ai-yoga-trainer", "/content/ai-yoga-trainer"):
        if os.path.exists(os.path.join(base, "ai", "pipeline.py")):
            return os.path.abspath(base)
    hits = glob.glob("**/ai/pipeline.py", recursive=True)
    return os.path.abspath(os.path.join(os.path.dirname(hits[0]), "..")) if hits else None

ROOT = find_root()
if ROOT is None:
    if "YOUR-USERNAME" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "ai-yoga-trainer"], check=True)
    else:
        from google.colab import files
        print("Upload ai-yoga-trainer.zip (your zipped project folder):")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall(".")
    ROOT = find_root()
assert ROOT, "Project not found - clone the repository or upload the zip."
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("Project root:", ROOT)

## 2. Install dependencies (Colab already has TensorFlow and NumPy)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

import numpy, pandas, cv2, matplotlib
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| opencv", cv2.__version__, "| matplotlib", matplotlib.__version__)
try:
    import tensorflow as tf
    print("tensorflow", tf.__version__)
except ImportError:
    print("TensorFlow not installed - run: pip install tensorflow-cpu  (needed for MoveNet)")

## 3. Load MoveNet
The model file is downloaded to `ai/models/` on first use. Commit this file to GitHub if you want the deployed app to start without downloading.

In [ ]:
import time
import numpy as np
from ai.models.registry import ensure_model, model_path
from ai.pose.movenet import MoveNetDetector

MODEL_VARIANT = "lightning"  #@param ["lightning", "thunder"]
path = ensure_model(MODEL_VARIANT)
detector = MoveNetDetector(MODEL_VARIANT, model_file=str(path))
print("Model file:", path, f"({path.stat().st_size / 1e6:.1f} MB), input size", detector.input_size)

blank = np.zeros((480, 640, 3), dtype=np.uint8)
detector.detect(blank)                                  # warm-up
t0 = time.time()
for _ in range(20):
    kps = detector.detect(blank)
print(f"Inference: {(time.time() - t0) / 20 * 1000:.1f} ms/frame on this runtime; keypoint array shape {kps.shape}")

## 4. Self-check with synthetic skeletons (no camera needed)
Idealised stick figures verify that the rule engine, classifier, scoring and corrections behave as designed. These are **not** real people.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from utils.synthetic import make_pose, to_pixels
from utils.plotting import skeleton_figure
from ai.pose.normalization import normalize_keypoints
from ai.pose.geometry import compute_metrics
from ai.yoga.classifier import classify

cases = [("T Pose", "both"), ("Tree Pose", "left"), ("Warrior II", "left"), ("Chair Pose", "both"), ("Triangle Pose", "right")]
fig, axes = plt.subplots(1, len(cases), figsize=(14, 3.6))
rows = []
for ax, (name, variant) in zip(axes, cases):
    coords = make_pose(name, variant)
    skeleton_figure(coords, name, ax=ax)
    norm = normalize_keypoints(to_pixels(coords, noise_px=1.0))
    result = classify(compute_metrics(norm.coords))
    rows.append({"Synthetic pose": name, "Classified as": result.pose, "Side": result.variant,
                 "Score": round(result.score), "Margin to runner-up": round(result.margin)})
plt.show()
pd.DataFrame(rows)

In [ ]:
# Explainable corrections on a deliberately flawed synthetic Warrior II (front knee too straight)
import numpy as np
import pandas as pd
from utils.synthetic import make_pose, to_pixels
from ai.pipeline import YogaPipeline
from ai.pose.keypoints import KP

flawed = make_pose("Warrior II", "left")
flawed[KP["left_ankle"]] = flawed[KP["left_knee"]] + np.array([0.45, 0.9])

pipe = YogaPipeline(pose="Warrior II", variant="left", difficulty="Intermediate")
for i in range(30):
    res = pipe.analyze_keypoints(to_pixels(flawed, noise_px=1.0), i / 15)
print("Message:", res.message, "\n")
for c in res.corrections:
    print(c.text)
pd.DataFrame([c.as_dict() for c in res.corrections])

## 5. Provide input: image, video or webcam snapshot
Pick **one** of the three cells below. Your pose should be filmed from the front (Chair Pose from the side) with your whole body visible.

In [ ]:
# 5a. Upload a video (MP4 recommended)
from google.colab import files
up = files.upload()
VIDEO_PATH = next(iter(up))
print("Video:", VIDEO_PATH)

In [ ]:
# 5b. Take a webcam snapshot (Colab asks for camera permission)
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="photo.jpg", quality=0.9):
    js = Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    """)
    display(js)
    data = eval_js("takePhoto({})".format(quality))
    with open(filename, "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    return filename

IMAGE_PATH = take_photo()

In [ ]:
# 5c. Or upload a still image
from google.colab import files
up = files.upload()
IMAGE_PATH = next(iter(up))

## 6. Analyse a single image

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ai.pipeline import YogaPipeline
from utils.drawing import annotate_frame
from utils.video import bgr_to_rgb, resize_max_side

TARGET_POSE = "Warrior II"   #@param ["T Pose", "Tree Pose", "Warrior II", "Chair Pose", "Triangle Pose"]
SIDE = "auto"                #@param ["auto", "left", "right"]
LEVEL = "Intermediate"       #@param ["Beginner", "Intermediate", "Advanced"]

img = resize_max_side(cv2.imread(IMAGE_PATH), 960)
pipe = YogaPipeline(detector, TARGET_POSE, SIDE if SIDE != "auto" else "auto", LEVEL)
res = pipe.process_frame(img, 0.0)
plt.figure(figsize=(7, 7)); plt.imshow(bgr_to_rgb(annotate_frame(img, res))); plt.axis("off"); plt.show()

print("Detected pose:", res.detected_pose, "| target class score:", round(res.class_score))
if res.detected:
    print(f"Overall {res.overall:.0f}  alignment {res.alignment.score:.0f}  stability {res.stability:.0f}")
    for c in res.corrections:
        print(" -", c.text)
else:
    print(res.message)

## 7. Analyse a video - the complete pipeline
Runs MoveNet on every sampled frame, then shows the skeleton, scores, hold timer, per-joint statistics and corrections.

In [ ]:
import numpy as np, pandas as pd
from utils.video import iter_video_frames, probe_video

TARGET_POSE = "Warrior II"   #@param ["T Pose", "Tree Pose", "Warrior II", "Chair Pose", "Triangle Pose"]
SIDE = "auto"                #@param ["auto", "left", "right"]
LEVEL = "Intermediate"       #@param ["Beginner", "Intermediate", "Advanced"]
MAX_SECONDS = 60             #@param {type:"integer"}

info = probe_video(VIDEO_PATH)
print(f"Video: {info.width}x{info.height}, {info.fps:.1f} fps, {info.duration_s:.1f}s")

pipe = YogaPipeline(detector, TARGET_POSE, SIDE, LEVEL, source="colab-video")
pipe.start_session("colab-video")
times, series, best, worst = [], {}, (None, -1), (None, 101)
for frame, t in iter_video_frames(VIDEO_PATH, target_fps=10, max_seconds=MAX_SECONDS):
    r = pipe.process_frame(frame, t)
    if not r.detected:
        continue
    times.append(t)
    for k, v in r.metrics.items():
        series.setdefault(k, []).append(v)
    img = bgr_to_rgb(annotate_frame(frame, r))
    if r.overall > best[1]: best = (img, r.overall)
    if r.overall < worst[1]: worst = (img, r.overall)

summary = pipe.recorder.summary()
print({k: summary[k] for k in ["pose", "variant", "difficulty", "duration_s", "detected_frames", "held_s",
                               "best_streak_s", "completed", "avg_score", "avg_alignment", "avg_stability", "avg_symmetry"]})

In [ ]:
from utils.plotting import timeline_figure, joint_error_figure, metric_timeseries_figure
import matplotlib.pyplot as plt

fig = timeline_figure(summary["timeline"], summary["hold_target_s"]); plt.show()
fig = joint_error_figure(summary["errors"]); plt.show()

# measured angles vs target bands for the (most recent) variant used
from ai.yoga.poses import criteria_for
crit = [c for c in pipe.criteria_for_variant(summary["variant"]) if c.is_angle and c.metric.endswith(("knee", "elbow", "shoulder"))][:4]
fig = metric_timeseries_figure(times, {c.label: series[c.metric] for c in crit}, {c.label: (c.low, c.high) for c in crit}); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (im, sc), title in zip(axes, (best, worst), ("Best moment", "Needs most work")):
    ax.imshow(im); ax.set_title(f"{title} (score {sc:.0f})"); ax.axis("off")
plt.show()

pd.DataFrame(summary["errors"])[["label", "frames_total", "bad_frac", "mean_deviation", "direction"]]

In [ ]:
# Feedback in words
from ai.feedback.recommendations import session_feedback
for line in session_feedback(summary):
    print("-", line)

## 8. Personalised calibration
Hold your most comfortable version of the pose; the target ranges are widened (never tightened, and only by a bounded amount) to fit you.

In [ ]:
from ai.personalization.calibration import apply_calibration, describe_adaptation
from ai.yoga.poses import criteria_for

CAL_VARIANT = summary["variant"]          # the side that was detected in section 7
cal = YogaPipeline(detector, TARGET_POSE, CAL_VARIANT, "Intermediate")
cal.start_calibration()
for frame, t in iter_video_frames(VIDEO_PATH, target_fps=10, max_seconds=MAX_SECONDS):
    cal.process_frame(frame, t)
profile = cal.calibrator.build_profile()
print("Calibration frames:", profile["n_samples"], "| proportions:", profile["proportions"])

base = criteria_for(TARGET_POSE, CAL_VARIANT)
pd.DataFrame(describe_adaptation(base, apply_calibration(base, profile)) or [{"Info": "Your values are inside the default ranges."}])

In [ ]:
# Re-run the video with the personalised ranges
pipe2 = YogaPipeline(detector, TARGET_POSE, CAL_VARIANT, LEVEL, calibration={CAL_VARIANT: profile}, source="colab-video")
pipe2.start_session("colab-video")
for frame, t in iter_video_frames(VIDEO_PATH, target_fps=10, max_seconds=MAX_SECONDS):
    pipe2.process_frame(frame, t)
s2 = pipe2.recorder.summary()
print(f"Without calibration: score {summary['avg_score']:.1f}, held {summary['held_s']:.1f}s")
print(f"With calibration   : score {s2['avg_score']:.1f}, held {s2['held_s']:.1f}s")

## 9. Session history, progress, recurring errors and recommendations (SQLite)
The database module is the same one the web app uses (SQLite here; PostgreSQL optional for deployment).

In [ ]:
from database.db import Database
from ai.personalization.progress import prepare_sessions, overall_summary, per_pose_summary
from ai.personalization.recurring import detect_recurring_errors
from ai.feedback.recommendations import build_recommendations
from utils.plotting import progress_figure, recurring_figure

db = Database("sqlite:///data/colab_demo.db")
uid = db.get_or_create_user("colab-user")
db.save_session(uid, summary)
db.save_session(uid, s2) if "s2" in globals() else None
print("Saved. Sessions stored:", len(db.get_sessions(uid)))

sessions = db.get_sessions(uid)
display(per_pose_summary(sessions))
fig = progress_figure(prepare_sessions(sessions)); plt.show()

rec = detect_recurring_errors(db.get_errors(uid))
print("Recurring errors need >= 2 sessions of the same pose with the same joint out of range.")
display(rec)
for r in build_recommendations(sessions, rec, db.get_levels(uid)):
    print(f"[{r.category}] {r.title}\n    {r.detail}\n    why: {r.reason}")

### 9b. Optional: SYNTHETIC history for demonstrating the dashboards
This generates *artificial* sessions from stick figures (with a deliberately recurring front-knee error). It is only for demonstrating recurring-error detection and recommendations - never present it as real user data.

In [ ]:
from datetime import datetime, timedelta, timezone
import numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from ai.pipeline import YogaPipeline
from ai.pose.keypoints import KP
from database.db import Database
from utils.synthetic import make_pose, to_pixels
from utils.plotting import progress_figure, recurring_figure
from ai.personalization.progress import prepare_sessions
from ai.personalization.recurring import detect_recurring_errors
from ai.feedback.recommendations import build_recommendations

demo_db = Database("sqlite:///:memory:")
demo_uid = demo_db.get_or_create_user("synthetic-demo")
rng = np.random.default_rng(0)
start = datetime.now(timezone.utc) - timedelta(days=8)
for day in range(6):
    coords = make_pose("Warrior II", "left")
    if day < 5:      # recurring error: front knee too straight in most sessions
        coords[KP["left_ankle"]] = coords[KP["left_knee"]] + np.array([0.45, 0.9])
    p = YogaPipeline(None, "Warrior II", "left", "Beginner", source="synthetic-demo")
    p.start_session("synthetic-demo")
    for i in range(10 * 15):
        p.analyze_keypoints(to_pixels(coords, noise_px=1.0 + (5 - day) * 0.3, rng=rng), i / 15)
    s = p.recorder.summary(); s["started_at"] = (start + timedelta(days=day)).isoformat(timespec="seconds")
    demo_db.save_session(demo_uid, s)

demo_sessions = demo_db.get_sessions(demo_uid)
rec = detect_recurring_errors(demo_db.get_errors(demo_uid))
display(rec[["pose", "label", "sessions_affected", "sessions_considered", "avg_bad_pct", "direction"]])
fig = recurring_figure(rec); plt.show()
fig = progress_figure(prepare_sessions(demo_sessions)); plt.show()
for r in build_recommendations(demo_sessions, rec, demo_db.get_levels(demo_uid))[:4]:
    print(f"[{r.category}] {r.title} - {r.reason}")

## 10. Run the unit tests (same tests as the repository)

In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, "-W", "ignore", "-m", "unittest", "discover", "-s", "tests", "-t", ".", "-v"],
                     capture_output=True, text=True)
print(out.stderr[-3000:])

## 11. Save the MoveNet model for GitHub / deployment
Download the model file, put it in `ai/models/` of your repository and commit it - the deployed Streamlit app then starts without downloading anything (about 5 MB for Lightning).

In [ ]:
from google.colab import files
files.download(str(ensure_model(MODEL_VARIANT)))